# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs, as described in the Croissant metadata. Entities are referenced by their `@id` fields.

In [ ]:
# List all record sets with their @id and descriptions

def print_record_sets(dataset):
    print("Available record sets:")
    for record_set in dataset.metadata.record_sets:
        print(f"- @id: {record_set.id}")
        print(f"  name: {record_set.name if hasattr(record_set, 'name') else ''}")
        print(f"  description: {record_set.description if hasattr(record_set, 'description') else ''}")

print_record_sets(dataset)

# List fields for each record set
for record_set in dataset.metadata.record_sets:
    print(f"\nFields for RecordSet @id: {record_set.id}")
    for field in record_set.fields:
        field_id = field.id
        print(f"  - @id: {field_id}")
        print(f"    name: {field.name if hasattr(field, 'name') else ''}")
        print(f"    description: {field.description if hasattr(field, 'description') else ''}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Collect all record set @ids
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Record set @ids: {record_sets_ids}")

# Load dataframes for all record sets
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records in record set {record_set_id}")

# Pick the main protein abundance table
# For this dataset, the main data is likely under a record set containing `protein_abundance_table`, but we must detect its @id.
main_record_set_id = None
for rs in dataset.metadata.record_sets:
    if hasattr(rs, 'name') and 'abundance' in (rs.name or '').lower():
        main_record_set_id = rs.id
        break
if not main_record_set_id:
    # If not matched by name, select the first record set
    main_record_set_id = record_sets_ids[0]
print(f"Selected main record set: {main_record_set_id}")

print("Columns in main record set:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All columns and fields are referenced by their schema `@id` values.

In [ ]:
# Choose a numeric field for EDA
# We'll attempt to find a suitable numeric field, such as 'abundance' or 'molecular weight', from field names
df = dataframes[main_record_set_id]
numeric_field_id = None
for col in df.columns:
    if any(x in col.lower() for x in ['abundance', 'mw', 'weight', 'count', 'peptide', 'coverage']):
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
# If none found, use first numeric
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
assert numeric_field_id is not None, "No numeric field found for EDA."
print(f"Using numeric field for analysis: {numeric_field_id}")

# Filtering on this numeric field
threshold = df[numeric_field_id].mean()  # Use mean value as threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalizing the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to find a grouping field (e.g. sample, treatment, or description)
group_field_id = None
for col in df.columns:
    if any(x in col.lower() for x in ['sample', 'treatment', 'group', 'condition', 'description']):
        if pd.api.types.is_string_dtype(df[col]) or df[col].nunique() < 20:
            group_field_id = col
            break
# If still none, skip grouping
if group_field_id:
    print(f"Grouping on field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the distribution of the selected numeric field, and a boxplot grouped by any available categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot grouped by possible group_field_id
if group_field_id:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, explore, and process the FAIR\u00b2 dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. We accessed the data by schema `@id`, inspected and filtered protein abundance information, normalized numeric values, and visualized distributions for protein features. This structured approach enables reproducible FAIR data handling for downstream analytical tasks.